In [1]:
import pandas as pd
import numpy as np

In [5]:
pip install --upgrade nbformat plotly

^C
Note: you may need to restart the kernel to use updated packages.


In [2]:
import plotly.express as pltexp

In [3]:
df = pd.read_pickle("df_unite.pkl")

In [4]:
df_date = [i.year for i in df["first_release_date"]]
df_year = pd.DataFrame(df_date, columns=['col_name'])
df_with_year = pd.merge(df, df_year, left_index=True, right_index=True, how='inner')
df_with_year = df_with_year.rename(columns={'col_name': 'year'})

В данном документе рассмотрим более детальную аналитику, которая уже напрямую позволит вывести допущения и бизнес-вывод. Нам, как аналитикам, очень интересно понять, какие вообще игры популярны сейчас. Может быть есть какой-то тренд на рост/падение интереса к определённым жанрам. Тогда, конечно, не стоит создавать такие игры

## График 1: Доля жанров по годам

In [5]:
box_genre = []

for elems in df.columns:
    if "genre" in elems:
        box_genre.append(elems)

In [6]:
df_by_years = df_with_year[df_with_year['year'].between(1995, 2025)]
df_by_years = df_by_years.groupby('year')[box_genre].sum()

In [7]:
df_by_years

,genre_Adventure,genre_Arcade,genre_Card & Board Game,genre_Fighting,genre_Hack and slash/Beat 'em up,genre_Indie,genre_MOBA,genre_Music,genre_Pinball,genre_Platform,...,genre_Racing,genre_Real Time Strategy (RTS),genre_Role-playing (RPG),genre_Shooter,genre_Simulator,genre_Sport,genre_Strategy,genre_Tactical,genre_Turn-based strategy (TBS),genre_Visual Novel
year,,,,,,,,,,,,,,,,,,,,,
1995.0,57,32,2,33,9,2,0,0,4,51,...,15,5,24,43,15,21,24,9,7,1
1996.0,66,38,0,33,8,4,0,1,0,29,...,19,9,36,47,18,26,36,7,15,4
1997.0,63,34,2,27,9,4,0,1,1,40,...,27,20,30,60,35,32,44,7,9,1
1998.0,93,26,5,35,11,9,0,3,1,32,...,40,21,40,55,42,36,48,9,7,0
1999.0,107,24,3,33,15,6,0,5,1,39,...,47,17,62,74,50,36,52,16,16,0
2000.0,86,23,4,23,20,5,0,3,2,40,...,42,21,44,58,51,38,65,13,18,3
2001.0,94,27,6,25,17,9,0,6,0,42,...,45,29,55,54,55,41,74,19,19,0
2002.0,130,23,6,30,19,12,0,5,1,60,...,39,25,69,75,65,73,75,28,18,4
2003.0,130,26,9,41,34,15,0,3,2,47,...,52,30,86,79,75,50,95,32,27,2


In [8]:
box_by_years = []
for elems in df_by_years.columns:
    box_by_years.append(elems.replace('genre_', ''))

In [9]:
df_by_years.columns = box_by_years

In [10]:
df_by_years = df_by_years.div(df_by_years.sum(axis=1), axis=0)

In [11]:
pltexp.area(df_by_years.reset_index(), x='year', y=df_by_years.columns, title='Доля жанров по годам')

## График 2: Средний рейтинг по годам  

Мы построили график, в котором можно посмотреть долю жанров по годам выхода различных игрушек. Теперь давайте в целом посмотрим, а как меняется рейтинг игр от года к году. Это, вместе с анализом рынка/кризисов, поможет нам более детально понимать, в какие периоды стоит выпускать много игр, а когда лучше придержать коней и доработать игру, не выпуская её в невыгодный для рынка момент

In [12]:
df_with_year.info()

<class 'pandas.DataFrame'>
RangeIndex: 23203 entries, 0 to 23202
Data columns (total 93 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   aggregated_rating                                     10274 non-null  float64
 1   aggregated_rating_count                               10274 non-null  float64
 2   first_release_date                                    23182 non-null  object 
 3   keywords                                              18391 non-null  object 
 4   name                                                  23203 non-null  str    
 5   rating                                                23203 non-null  float64
 6   rating_count                                          23203 non-null  int64  
 7   slug                                                  23203 non-null  str    
 8   total_rating                                          23203 non-nul

In [13]:
df_rating_by_year = df_with_year.groupby('year')['rating']
df_rating_by_year = df_rating_by_year.mean().reset_index()

In [14]:
df_rating_by_year

,year,rating
0,1952.0,31.793258
1,1958.0,62.227261
2,1962.0,68.138435
3,1968.0,39.775281
4,1971.0,49.776558
5,1972.0,78.862076
6,1973.0,54.053211
7,1974.0,58.426966
8,1975.0,60.804373
9,1976.0,65.222564


In [15]:
pltexp.line(df_rating_by_year, x='year', y='rating', title='Средний рейтинг по годам', markers=True)

## График 3: Зависимость рейтинга игр от продуктивности компании (с частоты выхода игр)

Посмотрим, насколько сильно продуктивность студии влияет на рейтинг выпущенных ими игр. Это позволит сделать ряд допущений и выводов о том, какую стратегию компаниям лучше выбирать - выпускать как можно больше игры или бить редко, но метко

In [16]:
df_developers_by_rating = df_with_year.groupby('developer')
df_developers_by_rating = df_developers_by_rating.agg(mean_rating=('rating','mean'), count=("rating","count"), year_min=("year","min"), year_max=("year","max")).reset_index()

In [17]:
df_developers_by_rating

,developer,mean_rating,count,year_min,year_max
0,,65.688572,2041,1968.0,2025.0
1,"(Archive) Capcom Co., LTD.",76.642036,2,2015.0,2024.0
2,",",74.000000,1,1990.0,1990.0
3,.GEARS Studios,47.806087,1,2013.0,2013.0
4,.theprodukkt,31.091217,1,2009.0,2009.0
...,...,...,...,...,...
10065,墨鱼玩游戏,68.312716,1,2018.0,2018.0
10066,拾英工作室,75.416027,1,2019.0,2019.0
10067,旋風野郎,76.404494,1,2019.0,2019.0
10068,蒸汽满满工作室,70.102925,1,2024.0,2024.0


In [18]:
df_developers_by_rating = df_developers_by_rating[df_developers_by_rating["count"] > 1]

In [19]:
df_developers_by_rating = df_developers_by_rating[df_developers_by_rating["count"] < 100]

In [20]:
df_developers_by_rating = df_developers_by_rating.sort_values("count").reset_index()

In [21]:
df_developers_by_rating

,index,developer,mean_rating,count,year_min,year_max
0,30,1C: Maddox Games,71.680420,2,2006.0,2011.0
1,157,A44,72.194169,2,2018.0,2024.0
2,169,AHEARTFULOFGAMES,71.509192,2,2016.0,2024.0
3,10019,thecatamites,72.707787,2,2010.0,2016.0
4,9950,nStigate Games,62.097789,2,2009.0,2012.0
...,...,...,...,...,...,...
2826,2480,EA Canada,72.391690,67,1994.0,2018.0
2827,5566,Namco,69.970841,68,1980.0,2007.0
2828,8588,Telltale Games,75.764207,86,2005.0,2019.0
2829,9108,Ubisoft Montreal,70.916923,87,1999.0,2022.0


In [22]:
box_min_year = [elem for elem in df_developers_by_rating["year_min"]]
box_max_year = [elem for elem in df_developers_by_rating["year_max"]]
box_count = [elem for elem in df_developers_by_rating["count"]]

In [23]:
box_mean_gap_time = []

In [24]:
box_mean_gap_time

[]

In [25]:
for i in range (0, len(box_min_year)):
    box_mean_gap_time.append((box_max_year[i] - box_min_year[i]) / box_count[i])

In [26]:
box_mean_gap_time = pd.DataFrame(box_mean_gap_time, columns=['gap_year'])

In [27]:
box_mean_gap_time

,gap_year
0,2.500000
1,3.000000
2,4.000000
3,3.000000
4,1.500000
...,...
2826,0.358209
2827,0.397059
2828,0.162791
2829,0.264368


In [28]:
df_developers_by_rating_with_mean_gap_time = pd.merge(df_developers_by_rating, box_mean_gap_time, left_index=True, right_index=True, how='inner')

In [29]:
df_developers_by_rating_with_mean_gap_time

,index,developer,mean_rating,count,year_min,year_max,gap_year
0,30,1C: Maddox Games,71.680420,2,2006.0,2011.0,2.500000
1,157,A44,72.194169,2,2018.0,2024.0,3.000000
2,169,AHEARTFULOFGAMES,71.509192,2,2016.0,2024.0,4.000000
3,10019,thecatamites,72.707787,2,2010.0,2016.0,3.000000
4,9950,nStigate Games,62.097789,2,2009.0,2012.0,1.500000
...,...,...,...,...,...,...,...
2826,2480,EA Canada,72.391690,67,1994.0,2018.0,0.358209
2827,5566,Namco,69.970841,68,1980.0,2007.0,0.397059
2828,8588,Telltale Games,75.764207,86,2005.0,2019.0,0.162791
2829,9108,Ubisoft Montreal,70.916923,87,1999.0,2022.0,0.264368


In [30]:
df_developers_by_rating_with_mean_gap_time = df_developers_by_rating_with_mean_gap_time[df_developers_by_rating_with_mean_gap_time["gap_year"] < 3]

In [31]:
fig = pltexp.scatter(df_developers_by_rating_with_mean_gap_time, x='count', y='mean_rating', color='gap_year', opacity=0.6, size='count', title='Зависимость рейтинга игр от продуктивности компании (с частоты выхода игр)',)
fig.show()

## График 4: Зависимость количества доступных платформ и рейтинга игр

На этапе выбора стратегии разработки игр очень часто в первую очередь встает вопрос - а как лучше, разработать игру под как можно большее количество платформ или остановиться на одной/нескольких и не распылятся? Для этого мы выделили отдельную сущность "available_platforms", которая и позволяет посмотреть количество доступных платформ для определенёной игры. Построим график.

In [32]:
pltexp.box(df_with_year, x='available_platforms', y='rating', title='Зависимость количества доступных платформ и рейтинга игр')

## График 5: Средний рейтинг по жанрам

Для того, чтобы выбрать жанр, помимо тренда, который мы анализировали выше, нам также необходимо понимать средний рейтинг в каждом жанре. Для этого построим соответствующий график

In [33]:
box_by_years = []

In [34]:
box_genre = []

for elems in df_with_year.columns:
    if "genre" in elems:
        box_genre.append(elems)
    

In [35]:
box_by_years = []

for elems in box_genre:
    genre = elems.replace('genre_', '')
    ratings = df_with_year.loc[df[elems] == 1, 'rating']
    box_by_years.append({'genre': genre, 'mean_rating': ratings.mean(), 'count': len(ratings)})

In [36]:
graph_rating_genre = pd.DataFrame(box_by_years)
graph_rating_genre = graph_rating_genre.sort_values('mean_rating')
pltexp.bar(graph_rating_genre, x='mean_rating', y='genre', title='Средний рейтинг по жанрам', hover_data=['count'])

##  График 6: Зависимость длины игры и её рейтинга

На начальном этапе разработки игры также важно понимать, какой примерно длины мы будем разрабатывать игру. Делать её более длинной или всё же короткой. Для того, чтобы сделать необходимые выводы, посмотрим, как же влияет длина игры на её рейтинг. Для этого построим соответствующий график

In [59]:
df_time_grop = df_with_year.dropna(subset=['avg_time'])

In [60]:
df_time_grop = df_with_year[df_with_year['avg_time'] <= 400] # убираю выбросы, которые дольше чем 400 часов

In [62]:

pltexp.scatter(df_time_grop, x='avg_time', y='rating', title='Зависимость длины игры и её рейтинга', )